# Challenge 1: Credit default

### Things to fix / add for Preprocessing part


- [ ] **Missing values** — add `df.isnull().sum()` to show you checked.
- [ ] **Class distribution** — add `value_counts()` + a bar plot. We need this for training.
- [ ] **Save the scaler** — add `joblib.dump(scaler, "scaler.pkl")` after fitting. `predict.py` needs it.
- [ ] **EDA** — add basic exploratory analysis (`.describe()`, feature distributions, correlations with target).
- [ ] **Justify choices** — fill in every `_Your notes:_` (bipolar vs binary, why PAY_* is numeric, why StandardScaler) as ittt  says in the assesment handout.


In [1]:
# --- imports ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Load data

_Your notes:_



In [ ]:
# first data row is headers -> header=1
# DATA_PATH = "default of credit card clients.xls"
DATA_PATH = "/content/drive/MyDrive/default of credit card clients.xls"

df = pd.read_excel(DATA_PATH, header=1)
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


## Clean categories

_Your notes:_



In [3]:
# drop ID; map rare EDUCATION/MARRIAGE codes to "other" (justify in markdown)
df = df.drop(columns=["ID"])
df["EDUCATION"] = df["EDUCATION"].replace([0, 5, 6], 4)
df["MARRIAGE"] = df["MARRIAGE"].replace(0, 3)
print(df[["EDUCATION", "MARRIAGE"]].head())

   EDUCATION  MARRIAGE
0          2         1
1          2         2
2          2         2
3          2         1
4          2         1


In [4]:
# quick peek (optional)
df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


## Features and target

_Your notes:_



In [5]:
X = df.drop(columns=["default payment next month"])
y = df["default payment next month"]

## Train / test split

_Your notes (e.g. stratify):_



In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (24000, 23) Test: (6000, 23)


## Categoricals (one-hot + bipolar)

_Your notes:_



In [7]:
cat_cols = ["SEX", "EDUCATION", "MARRIAGE"]

In [8]:
# one-hot on train and test separately
X_train_cat = pd.get_dummies(X_train[cat_cols], columns=cat_cols)
X_test_cat = pd.get_dummies(X_test[cat_cols], columns=cat_cols)
print("One-hot shape:", X_train_cat.shape)

One-hot shape: (24000, 9)


In [9]:
# same columns on test as train
X_train_cat, X_test_cat = X_train_cat.align(
    X_test_cat, join="left", axis=1, fill_value=0
)

In [10]:
# 0/1 -> -1/+1 for nominal dummies (explain why in markdown above)
X_train_bipolar = X_train_cat.astype(float).replace({0: -1.0})
X_test_bipolar = X_test_cat.astype(float).replace({0: -1.0})

In [11]:
print("Bipolar shape:", X_train_bipolar.shape)
print("Sample:", X_train_bipolar.iloc[0].values[:5], "...")

Bipolar shape: (24000, 9)
Sample: [-1.  1. -1.  1. -1.] ...


## Numeric columns — scale

_Your notes:_



In [12]:
# everything except SEX, EDUCATION, MARRIAGE (incl. PAY_*, bill/pay amounts, AGE, LIMIT_BAL)
num_cols = [col for col in X.columns if col not in cat_cols]
print("Scaling", len(num_cols), "columns")

Scaling 20 columns


In [13]:
# fit scaler on train only
scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(X_train[num_cols])
X_test_num_scaled = scaler.transform(X_test[num_cols])
print("Mean (train, ~0):", f"{X_train_num_scaled.mean():.4f}")
print("Std  (train, ~1):", f"{X_train_num_scaled.std():.4f}")

Mean (train, ~0): -0.0000
Std  (train, ~1): 1.0000


## Final design matrix

_Your notes (column order for predict.py):_



In [14]:
# [scaled numerics | bipolar categoricals] — keep this order in inference
X_train_final = np.hstack([X_train_num_scaled, X_train_bipolar.values])
X_test_final = np.hstack([X_test_num_scaled, X_test_bipolar.values])

print("Train:", X_train_final.shape, "Test:", X_test_final.shape)
print("Num cols:", X_train_num_scaled.shape[1], "+ cat:", X_train_bipolar.shape[1])
print("First row:", X_train_final[0])

Train: (24000, 29) Test: (6000, 29)
Num cols: 20 + cat: 9
First row: [-0.05686623 -0.26455769  1.79331103  1.78019315  2.65204636  1.91181115
  0.2402604   0.25608731  1.50554693  1.74508934  1.77886926  1.89167911
  2.02083925  2.09634558  0.58065737 -0.29033241 -0.29781997  0.08696116
  0.50039738  0.04874486 -1.          1.         -1.          1.
 -1.         -1.         -1.          1.         -1.        ]


## Next — model training

_Your notes:_

